# HandTrace · egocentric factory video → hands, contacts, task steps, insights (PS1 + PS2 + PS3 + PS5)

Reproducibility and demo notebook for the Egocentric Vision Hackathon. All code lives in the `handtrace` package of the GitHub repository; this notebook clones it, sets the runtime options and calls the package. Every stage reads the previous stage's files, so the
problem statements are **linked, not just stacked**:

```
clip ─► PS1 WiLoR hands (21 kp, L/R, IDs) ─► post-processing (wearer filter, side vote, relink, gap fill) ─► hands_final.jsonl
            │                                                                                                  │
            │                         PS2 OWLv2 objects + fingertip rule + grab/release state machine ◄────────┘ ─► contacts.csv
            │                                                         │  object names = prompt hints             │ grab / release moments
            └────────── PS3 Gemini timeline (windows, fixed verbs, JSON schema) ◄─┘  ─► refine edges with PS2 ◄────┘ ─► actions_final.csv
                                                                                                                  │
PS5: quality filter · safety stretches (PS1 kp × PS2 hazard boxes) · plain-English search · dashboard · training-video cutter
Manual review (all stages on the original video): frame inspector widget · HTML player · audit → real accuracy numbers
```

| stage | output per clip (`OUT_ROOT/<clip>/`) | models / tools | needs |
|---|---|---|---|
| **PS1** | `hands_final.jsonl` (brief format + `id`, `kp3d`, `interpolated`, `side_corrected`) | official WiLoR detector + WiLoR 3D reconstruction (pinned commit) | GPU, MANO |
| **PS2** | `ps2_final/contacts.csv` (`clip, t_start, t_end, hand, state, object_name`, …) | OWLv2 open-vocabulary detector + fingertip rule | GPU |
| **PS3** | `ps3/actions_final.csv` (`clip, start_s, end_s, verb, object, hand`) | Gemini video API (auto-selects the current Flash model) + PS2 refinement | `GEMINI_API_KEY` |
| **PS5** | `ps5/quality.json`, `ps5/safety_events.csv`, `ps5_dashboard.html`, search, training video | OpenCV + the files above | CPU |
| **Review** | `review/manual_audit.csv`, `review/<clip>_review.html`, demo MP4s | ipywidgets inspector, standalone HTML player | — |

**Before you run**
1. Settings → Accelerator **GPU T4 x2** (one GPU also works), Internet **On**.
2. Add the clips as a **private** dataset, and `MANO_RIGHT.pkl` (mano.is.tue.mpg.de, after accepting the licence) as a second **private** dataset.
3. The repository must be reachable: set `REPO_URL` in section 1 (public GitHub URL, or a private Kaggle dataset folder holding a copy of the repo).
4. For PS3: Add-ons → Secrets → `GEMINI_API_KEY` (free key from Google AI Studio). For the `sam3` object backend also `HF_TOKEN`.
5. Edit section 1 (switch stages on/off with `RUN_PS1 … RUN_PS5`), then **Run All**. Every long stage is resumable: re-running skips finished clips.

**Data rules** (breaking them means disqualification): use the footage only for the hackathon; never post clips, frames or screenshots; demo videos go
only to the event's Drive folder; delete every copy (Kaggle datasets, outputs, notebook versions, Gemini file storage) when the event ends (section 14).
This notebook ships with **no outputs**; clear outputs again before sharing it. The HTML review player and the dashboard contain numbers only.

Numbers printed by the pipeline are **coverage and speed**. **Accuracy** comes only from frames you check by hand (sections 10 and 12).

## 1 · Settings

In [ ]:
# =============================================================  1 · SETTINGS  (the only cell you normally edit)
import time
SESSION_T0 = time.time()

# --- which stages run in this session ------------------------------------------------------------------------
RUN_PS1 = True                       # WiLoR hand tracking (GPU). False = reuse PS1 outputs already in OUT_ROOT
RUN_PS2 = True                       # objects + contacts (GPU). Needs PS1 outputs
RUN_PS3 = True                       # Gemini task-step timeline. Needs GEMINI_API_KEY; PS1/PS2 optional (used when present)
RUN_PS5 = True                       # quality / safety / search / dashboard / training clips (CPU)

# --- data ------------------------------------------------------------------------------------------------
CLIP_ROOTS = ["/kaggle/input"]       # where the private clips are mounted (duplicate file names: the largest copy wins)
DEMO_CLIPS = []                      # name parts of the demo clips, e.g. ["000003-004", "SGF00106"]: processed first, rendered in section 11

# --- PS1: hand tracking ----------------------------------------------------------------------------------
MODE = "auto"                        # "auto" | "fp32" | "fp16" | "fast"   (section 4 explains)
MAX_MEAN_PX, MAX_P95_PX = 1.0, 3.0   # MODE="auto": a faster mode must stay within this keypoint tolerance of fp32 (pixels)
DET_IMGSZ = 640                      # detector input size; section 4 measures 640 / 960 / 1280 on your clip (an earlier
                                     # run found 1280 detected ~40% more hands than 640 at ~4x detector time; WiLoR dominates the cost)
HANDEDNESS_SWAP = False              # True only if the section-5 sheet shows left/right systematically swapped
FRAME_STEP = 1                       # 1 = every frame; 2 = every 2nd frame + interpolation (about 2x faster, less accurate)
WORKERS = None                       # None = one worker per GPU
POST = {}                            # post-processing overrides, e.g. {"max_hands_per_side": 1}

# --- PS2: hand-object contact ----------------------------------------------------------------------------
OBJ_BACKEND = "owlv2"                # "owlv2" | "sam3" (gated: needs an HF_TOKEN Kaggle secret) | "yolo"
PS2_PROFILE = "auto"                 # object vocabulary: "auto" (picked from the clip name) or a profile in configs/ps2_objects.json
PS2 = dict(object_conf=0.15, object_imgsz=960, det_batch=4, det_every=1,        # det_every=2: detect every 2nd frame (faster)
           fingertip_threshold=2, pad_frac=0.15, min_contact_frames=5, release_grace_frames=3)

# --- PS3: task-step timeline (Gemini) ----------------------------------------------------------------------
PS3_MODEL = "auto"                   # "auto" = newest plain gemini-<version>-flash in the live list (gemini-3.5-flash is
                                     # deprecated and routed to gemini-3.6-flash as of Oct 2026); or an exact model id
PS3_FPS = 2.0                        # frames per second Gemini sees (API default 1: too slow for hands)
PS3_THINKING = "medium"              # minimal | low | medium | high | None
PS3_MEDIA_RES = None                 # None | "low" (4x fewer tokens per frame) | "medium" | "high"
PS3_SMALL_COPY_ROOTS = []            # folders holding clips_480p/ copies: uploaded instead of full resolution (same timeline)
EXPECTED_STEPS = []                  # optional reference task, e.g. ["pick screwdriver", "fasten screw", "place screwdriver"]

# --- PS5 ---------------------------------------------------------------------------------------------------
HAZARD_WORDS = None                  # None = default list (machine, spindle, lathe, blade, saw, press, conveyor, ...); or your own list

# --- run control -------------------------------------------------------------------------------------------
SESSION_LIMIT_H = 11.0               # never start a clip that could not finish inside this many hours (check Kaggle's current limit)
FORCE = []                           # ["post"] redo post-processing | ["ps1"] redo tracking | ["ps2"] redo object detection
WORK = "/kaggle/working/handtrace_run"
REPO_URL = "https://github.com/Avneetbhatia12/BiosHack.git"   # or "/kaggle/input/<PRIVATE-REPO-COPY>"
CODE = "/kaggle/working/HandTrace"         # the repository is cloned / copied here
OUT_ROOT = f"{WORK}/outputs"

## 2 · Environment

Kaggle's PyTorch / torchvision / numpy / OpenCV are **kept as they are** (every `pip install` runs under a constraints file),
so CUDA is never broken. `chumpy` is not installed (it fails on current numpy); MANO is converted once in section 3.
The Gemini key is read from Kaggle Secrets (or Colab userdata / the environment) into an environment variable and never printed.

In [ ]:
import os, sys, json, glob, shutil, subprocess
for d in (WORK, OUT_ROOT):
    os.makedirs(d, exist_ok=True)
if not os.path.isdir(f"{CODE}/handtrace"):
    if os.path.isdir(REPO_URL):
        shutil.copytree(REPO_URL, CODE, dirs_exist_ok=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, CODE], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)

from handtrace import environment as env
sh, get_secret = env.sh, env.get_secret          # used by later cells
print("python", sys.version.split()[0]); sh("nvidia-smi -L", check=False)
print(json.dumps(env.install_dependencies(ps1=RUN_PS1, ps2=RUN_PS2, ps3=RUN_PS3), indent=1))
if RUN_PS3:
    print("GEMINI_API_KEY is set:", env.load_secret("GEMINI_API_KEY"))   # the value is never printed
import torch
print("torch", torch.__version__, "| CUDA build", torch.version.cuda, "| cuda available", torch.cuda.is_available(),
      "| transformers", env.pkg_version("transformers"))

## 3 · Model assets

The project code is the `handtrace` package in the repository (no code is generated by this notebook). This cell fetches
the official WiLoR repository at a **pinned commit**, its pretrained weights, and prepares MANO (real `chumpy` at a pinned
commit, with an automatic chumpy-free fallback that repairs flat `shapedirs` arrays). Only needed for PS1.

| package | what it is |
|---|---|
| `handtrace/wilor/` | PS1 core: WiLoR detector, tracker, reconstruction, One-Euro smoothing, metrics, asset setup |
| `handtrace/postprocess.py`, `runner.py`, `render.py`, `results.py`, `cli.py` | post-processing, resumable multi-GPU runner, rendering, result loaders, CLI |
| `handtrace/ps2/`, `configs/ps2_objects.json` | PS2: OWLv2 / SAM 3 / YOLO objects, object tracker, fingertip contact rule, state machine |
| `handtrace/ps3/` | PS3: Gemini client, windows, prompt, validation, merge, PS2 refinement, worker comparison |
| `handtrace/ps5/` | PS5: event index, plain-English search, quality filter, safety monitor, dashboard, training video |
| `handtrace/review*.py`, `eval/` | manual review (frame composition, inspector widget, HTML player, audit) and scoring |

In [ ]:
# WiLoR repository (pinned), pretrained weights, MANO  (only needed for PS1)
WILOR = f"{WORK}/WiLoR"
WEIGHTS_DATASET_DIR = None            # offline: attach the weights as a private dataset, e.g. "/kaggle/input/wilor-weights"
if RUN_PS1:
    from handtrace.wilor.setup_utils import prepare_wilor
    status = prepare_wilor(WILOR, mano_search_roots=["/kaggle/input"], weights_dir=WEIGHTS_DATASET_DIR,
                           work_dir="/kaggle/working")
    print(json.dumps(status, indent=1))
else:
    print("RUN_PS1 = False: WiLoR assets skipped")

In [ ]:
# Clips, settings, early validation
import dataclasses
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Video
from handtrace import runner

def show(msg):                       # progress that refreshes in place
    from IPython.display import clear_output
    clear_output(wait=True); print(msg)

CLIPS = runner.discover_clips(CLIP_ROOTS)
assert CLIPS, f"no videos under {CLIP_ROOTS}: attach the private clips dataset (Add Input)"
ORDERED = runner.order_clips(CLIPS, DEMO_CLIPS)
STEMS = [runner.stem_of(c) for c in ORDERED]
N_GPU = runner.gpu_count(); N_WORKERS = WORKERS or max(1, N_GPU)
is_demo = lambda c: any(k.lower() in os.path.basename(c).lower() for k in DEMO_CLIPS)
print(f"{len(ORDERED)} clips | {N_GPU} GPU(s) -> {N_WORKERS} worker(s) | demo clips first: {[os.path.basename(c) for c in ORDERED if is_demo(c)]}")
display(pd.DataFrame({"clip": [os.path.basename(c) for c in ORDERED],
                      "MB": [round(os.path.getsize(c) / 1e6, 1) for c in ORDERED], "demo": [is_demo(c) for c in ORDERED]}))

if RUN_PS2 and OBJ_BACKEND == "sam3" and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = get_secret("HF_TOKEN") or ""          # Add-ons -> Secrets -> HF_TOKEN (never paste a token into a cell)
if RUN_PS2 and OBJ_BACKEND == "owlv2":                             # fetch the weights once so workers do not download in parallel
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(runner.ps2_config(runner.Settings(clips=ORDERED), ORDERED[0]).owl_model)
    except Exception as e:
        print("could not pre-download OWLv2:", repr(e))

S = runner.Settings(
    clips=ORDERED, out_root=OUT_ROOT, wilor_root=WILOR, prompts_path=f"{CODE}/configs/ps2_objects.json",
    priority=DEMO_CLIPS, mode="fp32" if MODE == "auto" else MODE, det_imgsz=DET_IMGSZ,
    handedness_swap=HANDEDNESS_SWAP, frame_step=FRAME_STEP, post=POST,
    ps2={**PS2, "backend": OBJ_BACKEND, "profile": PS2_PROFILE},
    deadline=SESSION_T0 + SESSION_LIMIT_H * 3600, force=FORCE, force_since=SESSION_T0)   # FORCE redoes only older outputs
runner.ps1_config(S, ORDERED[0]); runner.ps2_config(S, ORDERED[0])      # a typo in the settings fails here, not hours later
SETTINGS_JSON = f"{WORK}/settings.json"; S.save(SETTINGS_JSON)
json.dump({"gpus": runner.gpu_names(), "workers": N_WORKERS, "torch": torch.__version__, "cuda_build": torch.version.cuda,
           "stages": {"ps1": RUN_PS1, "ps2": RUN_PS2, "ps3": RUN_PS3, "ps5": RUN_PS5}},
          open(f"{OUT_ROOT}/environment.json", "w"), indent=1)       # for the efficiency slide

## 4 · PS1 speed and recall choices: precision mode and detector size

**Precision.** WiLoR's own `--fast` mode combines FP16 with skipping ViT blocks; skipping moved keypoints by ~3.6–4.2 px in earlier runs, so it
is only used if it passes the tolerance. This cell runs the same detections and crops through FP32, FP16-only and `--fast`, reports speed and the
keypoint deviation from FP32, and (with `MODE="auto"`) uses the fastest mode inside `MAX_MEAN_PX` / `MAX_P95_PX`, otherwise FP32.

**Detector size.** Hand-tracking recall is 30 % of the score. The second cell counts boxes at 640 / 960 / 1280 on the same frames. More boxes is
only better if they are real wearer hands: confirm on the section-5 sheets, then set `DET_IMGSZ` in section 1 and re-run from there.

In [ ]:
BENCH_START_FRAME = 600       # a moment with hands in view; 150 frames are measured. Change it if the report says no hands were found
RERUN_BENCH = False
bench_path = f"{OUT_ROOT}/benchmark_modes.json"
if RUN_PS1 and MODE == "auto":
    if os.path.exists(bench_path) and not RERUN_BENCH:
        BENCH = json.load(open(bench_path)); print("reusing", bench_path)
    else:
        from handtrace.wilor.pipeline import benchmark_modes
        cfg = dataclasses.replace(runner.ps1_config(S, ORDERED[0]), start_frame=BENCH_START_FRAME, max_frames=None)
        BENCH = benchmark_modes(cfg, n_frames=150, max_mean_px=MAX_MEAN_PX, max_p95_px=MAX_P95_PX, log=lambda *a: None)
        json.dump(BENCH, open(bench_path, "w"), indent=1)
    display(pd.DataFrame({m: BENCH[m] for m in ("fp32", "fp16", "fast") if m in BENCH}).T)
    if BENCH["hands"] == 0:
        print("WARNING: no hand in the benchmark frames, so nothing could be compared -> fp32. Try another BENCH_START_FRAME.")
    S.mode = BENCH["recommended"]
S.save(SETTINGS_JSON)
print("precision for the full run:", S.mode)

In [ ]:
DET_SIZE_CHECK = True         # boxes found per detector input size, on 50 frames of the first clip (about a minute)
if RUN_PS1 and DET_SIZE_CHECK:
    import gc
    from handtrace.wilor.wilor_detector import build_detector
    from handtrace.wilor.video_loader import VideoReader
    _base = runner.ps1_config(S, ORDERED[0]).resolved()
    _sample = [im for _, _, im in VideoReader(ORDERED[0], BENCH_START_FRAME, 150)][::3]
    _rows = []
    for sz in (640, 960, 1280):
        det = build_detector(dataclasses.replace(_base, det_imgsz=sz), "cuda" if torch.cuda.is_available() else "cpu")
        det(_sample[:2]); t0 = time.perf_counter(); n = 0; n_frames_with = 0
        for k in range(0, len(_sample), 8):
            for d in det(_sample[k:k + 8]):
                n += len(d); n_frames_with += int(len(d) > 0)
        dt = time.perf_counter() - t0
        _rows.append({"det_imgsz": sz, "boxes": n, "frames_with_a_box": n_frames_with, "frames": len(_sample),
                      "detector_ms_per_frame": round(1000 * dt / len(_sample), 1)})
        del det; gc.collect(); torch.cuda.empty_cache()
    display(pd.DataFrame(_rows).set_index("det_imgsz"))
    print(f"current DET_IMGSZ = {DET_IMGSZ}. WiLoR costs ~90-100 ms per hand on a T4, so the detector size is a small share of the total.")

## 5 · Smoke test (300 frames of one clip, PS1 then PS2)

Runs the **same worker code path** as the full run on the first 300 frames, so installation, weights, MANO, the object detector and the
outputs are all proven in minutes instead of after hours. Then look at the sheets: the worker's **left** hand must be labelled `L`.
Only if it is *systematically* swapped, set `HANDEDNESS_SWAP = True` in section 1 and re-run from section 1.

In [ ]:
SMOKE = dataclasses.replace(S, out_root=f"{WORK}/smoke", clips=[ORDERED[0]], max_frames=300, deadline=None, force=[])
if RUN_PS1:
    shutil.rmtree(SMOKE.out_root, ignore_errors=True)
    for stage in ("ps1", "ps2") if RUN_PS2 else ("ps1",):
        r = runner.launch(stage, SMOKE, f"{WORK}/settings_smoke.json", CODE, workers=1, poll_s=15, show=show)
        assert all(c == 0 for c in r["returncodes"]), f"{stage} failed (error above; full log: {r['logs'][0]})"
    display(pd.DataFrame(runner.write_summary(SMOKE.out_root, SMOKE.clips)).T)

In [ ]:
# Private review sheets (contain footage: clear this output before sharing the notebook)
from handtrace.render import review_sheet, ps2_sheet
from handtrace.wilor.evaluation import load_jsonl
clip = SMOKE.clips[0]; d = f"{SMOKE.out_root}/{runner.stem_of(clip)}"
if os.path.exists(f"{d}/hands_final.jsonl"):
    hands = load_jsonl(f"{d}/hands_final.jsonl")
    sheet = review_sheet(clip, hands, n=6, min_hands=2)
    if sheet is not None:
        plt.figure(figsize=(18, 10)); plt.imshow(sheet[:, :, ::-1]); plt.axis("off"); plt.title("PS1: is the worker's LEFT hand labelled L?"); plt.show()
    print("handedness spatial consistency (heuristic, left wrist left of right wrist):",
          json.load(open(f"{d}/metrics.json"))["handedness_spatial_consistency"])
    if os.path.exists(f"{d}/ps2_final/contacts.jsonl"):
        objs, cons = load_jsonl(f"{d}/ps2_final/object_detections.jsonl"), load_jsonl(f"{d}/ps2_final/contacts.jsonl")
        sheet2 = ps2_sheet(clip, hands[:len(objs)], objs, cons, n_pos=6, n_neg=3)
        if sheet2 is not None:
            plt.figure(figsize=(18, 12)); plt.imshow(sheet2[:, :, ::-1]); plt.axis("off"); plt.title("PS2: POS = contact, NEG = hand without contact"); plt.show()
        display(pd.read_csv(f"{d}/ps2_final/contacts.csv"))

## 6 · Full run: PS1 then PS2 on every clip

First a time projection from the smoke test (it includes model warm-up, so it is slightly pessimistic). Then PS1 on every clip, then PS2.
Re-running a stage continues where it stopped; `SESSION_LIMIT_H` stops new clips from starting if they could not finish. Save a notebook version
(or let Kaggle commit) so `/kaggle/working` outputs survive the session.

In [ ]:
from handtrace.wilor.video_loader import probe
frames_total = sum(probe(c).frame_count for c in ORDERED)
sd = f"{SMOKE.out_root}/{runner.stem_of(SMOKE.clips[0])}"
try:
    m1 = json.load(open(f"{sd}/metrics.json")); m2 = json.load(open(f"{sd}/ps2_final/ps2_metrics.json")) if RUN_PS2 else {"timing": {}}
except (NameError, FileNotFoundError):
    print("no smoke-test numbers (section 5 was skipped): no projection. The full run still works.")
else:
    fps1, fps2 = m1["end_to_end_fps"], (m2["timing"].get("object_pass_fps") or 1.0)
    h1 = runner.project_hours(fps1, 1, frames_total, N_WORKERS); h2 = runner.project_hours(fps2, 1, frames_total, N_WORKERS) if RUN_PS2 else 0
    print(f"{len(ORDERED)} clips, {frames_total} frames | PS1 {fps1:.1f} fps/worker -> {h1:.1f} h | PS2 {fps2:.1f} fps/worker -> {h2:.1f} h "
          f"| total {h1 + h2:.1f} h with {N_WORKERS} worker(s), budget {SESSION_LIMIT_H} h")
    if h1 + h2 > SESSION_LIMIT_H:
        print("Over budget. Levers, in order of how little accuracy they cost: MODE='auto'/'fp16' (section 4), a second GPU, "
              "PS2 det_every=2, DEMO_CLIPS (they run first), FRAME_STEP=2. Clips that do not fit simply stay pending for the next session.")
    longest = max(probe(c).frame_count for c in ORDERED)
    S.est_clip_s = {"ps1": longest / max(fps1, 1e-9), "ps2": longest / max(fps2, 1e-9)}   # priors for the time-budget check
    S.save(SETTINGS_JSON)

In [ ]:
# PS1 on every clip (resumable). Progress refreshes in place; logs are in OUT_ROOT/_logs
if RUN_PS1:
    res_ps1 = runner.launch("ps1", S, SETTINGS_JSON, CODE, workers=WORKERS, poll_s=60, show=show)

In [ ]:
# PS2 on every clip whose PS1 is finished (resumable)
if RUN_PS2:
    res_ps2 = runner.launch("ps2", S, SETTINGS_JSON, CODE, workers=WORKERS, poll_s=60, show=show)

## 7 · PS1 + PS2 results

Per-clip table, one combined `contacts_all.csv` (columns start with the brief's `clip, t_start, t_end, hand, state, object_name`) and a
grab/release timeline for the first clip.

In [ ]:
rows = runner.write_summary(OUT_ROOT, ORDERED)
print(runner.collect_contacts(OUT_ROOT, ORDERED), "contact intervals in contacts_all.csv")
display(pd.DataFrame(rows).set_index("clip"))

stem = runner.stem_of(ORDERED[0]); cpath = f"{OUT_ROOT}/{stem}/ps2_final/contacts.csv"
if os.path.exists(cpath) and len(pd.read_csv(cpath)):
    tl = pd.read_csv(cpath)
    fig, ax = plt.subplots(figsize=(16, 1 + 0.5 * tl["hand_id"].nunique()))
    colours = {"portable": "tab:orange", "fixed": "tab:blue", "unknown": "tab:gray"}
    for k, (hid, g) in enumerate(tl.groupby("hand_id")):
        ax.broken_barh([(a, b - a) for a, b in zip(g.t_start, g.t_end)], (k - 0.4, 0.8), facecolors=[colours[t] for t in g.object_type])
    ax.set_yticks(range(tl["hand_id"].nunique())); ax.set_yticklabels([f"hand {h}" for h in sorted(tl.hand_id.unique())])
    ax.set_xlabel("seconds"); ax.set_title(f"{stem}: contact intervals (orange portable, blue fixed, grey unknown)"); plt.show()

## 8 · PS3 task-step timeline (Gemini video understanding, refined by PS2)

Per clip: overlapping 60 s windows over the **original** timeline → Gemini (`VideoMetadata` start/end offsets, `PS3_FPS` frames per second,
fixed verb vocabulary enforced by a JSON response schema) → validation (time base, window bounds, verb / hand normalisation) → merge across
windows → `ps3/actions.json`. Then the **PS2 link**: action edges snap to the exact grab / release frames for the same hand and object
(within 1.5 s), unknown hands / objects are filled from the overlapping contact → `ps3/actions_final.csv` (the file to use).

* Model: `PS3_MODEL="auto"` picks the newest plain Flash model in the key's live list. Gemini 3.x guidance: leave temperature at the default,
  set `thinking_level` instead. Both are configurable; unsupported options are dropped automatically with a log line.
* Upload: a 480p copy with the same frame timing (from `PS3_SMALL_COPY_ROOTS`, else made by ffmpeg) — faster, under the free-tier 2 GB limit,
  and deleted again when the clip is done.
* Never pay twice: every reply is cached in `ps3/raw/`; a finished clip is skipped; the uploaded video is deleted from Gemini storage per clip.
* PS2 object names (when present) go into the prompt as *hints* only.

In [ ]:
# PS3 settings and API check
from handtrace import ps3
PS3 = ps3.PS3Config(model=PS3_MODEL, fps=PS3_FPS, thinking_level=PS3_THINKING, media_resolution=PS3_MEDIA_RES,
                    small_copy_roots=PS3_SMALL_COPY_ROOTS, window_s=60, overlap_s=10, time_base="auto",
                    use_ps2_hints=True, refine_with_ps2=True, snap_tol_s=1.5)
PS3_CLIP = ORDERED[0]                                  # demo clips come first in ORDERED; or set a path
if RUN_PS3:
    assert os.environ.get("GEMINI_API_KEY"), "add the GEMINI_API_KEY secret (Add-ons -> Secrets) and re-run section 2"
    from handtrace.ps3.gemini_client import GeminiVideoClient
    _gc = GeminiVideoClient(PS3)
    flash = sorted(n for n in (_gc.live_models() or []) if "flash" in n)
    print("model chosen:", _gc.model, "| flash models available to this key:", flash[:12])
print("PS3 clip:", os.path.basename(PS3_CLIP))

In [ ]:
# Smoke test: ONE window (the first 60 s) of one clip -> OUT_ROOT/<stem>/ps3/partial/
if RUN_PS3:
    r = ps3.process_clip(PS3_CLIP, OUT_ROOT, dataclasses.replace(PS3, only_windows=[0]), client=_gc)
    stem = runner.stem_of(PS3_CLIP)
    display(pd.read_csv(f"{OUT_ROOT}/{stem}/ps3/partial/actions.csv"))

In [ ]:
# Time-base check: window 1 (starts at 50 s). Read the log line "time base ...":
#   "absolute(auto)" -> the model returned times from the start of the full video (what the prompt asks for)
#   "relative(auto)" -> it returned times from the window start; the code added the offset. Then set time_base="relative".
# Check 3 actions by eye in the inspector (section 10) to confirm the times land on the right moments.
if RUN_PS3:
    r = ps3.process_clip(PS3_CLIP, OUT_ROOT, dataclasses.replace(PS3, only_windows=[1]), client=_gc)
    display(pd.read_csv(f"{OUT_ROOT}/{stem}/ps3/partial/actions.csv"))

In [ ]:
# Every clip, one at a time, resumable (windows done above are reused from ps3/raw: no repeat API calls).
# Writes ps3/actions.json + actions.csv (model) and actions_final.* (PS2-refined), then deletes the upload from Gemini storage.
if RUN_PS3:
    res_ps3 = ps3.run_all(ORDERED, OUT_ROOT, PS3)
    display(pd.DataFrame([{k: v for k, v in r.items() if k != "actions"} | {"actions": len(r.get("actions", []))} for r in res_ps3]))
else:                                                   # PS3 ran earlier: just (re)apply the PS2 refinement, CPU only
    for c in ORDERED:
        ps3.finalize_clip(OUT_ROOT, runner.stem_of(c), PS3)
print(ps3.collect_actions(OUT_ROOT, ORDERED), "rows in", f"{OUT_ROOT}/actions_all.csv")

In [ ]:
# PS3 next to PS1 / PS2 for one clip: what the model said, what PS2 changed, and what PS2 saw that no step explains
from handtrace import results as R
stem = runner.stem_of(PS3_CLIP)
doc = R.read_json(f"{OUT_ROOT}/{stem}/ps3/actions_final.json")
if doc:
    acts = pd.DataFrame(doc["actions"])
    display(acts[[c for c in ("start_s", "end_s", "verb", "object", "hand", "confidence", "refined") if c in acts]])
    print("refinement:", {k: v for k, v in doc["refinement"].items() if k != "portable_contacts_without_action"})
    miss = doc["refinement"].get("portable_contacts_without_action", [])
    if miss:
        print(f"{len(miss)} PS2 grab->release intervals with no PS3 step (possible missed steps):"); display(pd.DataFrame(miss).head(15))
    fig, ax = plt.subplots(figsize=(16, 3))
    con = R.load_contacts(OUT_ROOT, stem)
    for y, side in ((2, "left"), (1, "right")):
        ax.broken_barh([(c["t_start"], c["t_end"] - c["t_start"]) for c in con if c.get("hand") == side], (y - 0.35, 0.7), color="tab:orange")
    cmap = plt.get_cmap("tab20")
    verbs = sorted(acts.verb.unique())
    for _, a in acts.iterrows():
        ax.broken_barh([(a.start_s, a.end_s - a.start_s)], (-0.35, 0.7), color=cmap(verbs.index(a.verb) % 20))
        ax.text(a.start_s, 0.45, a.verb, fontsize=7, rotation=45)
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["PS3 steps", "PS2 right", "PS2 left"]); ax.set_xlabel("seconds")
    ax.set_title(f"{stem}: PS3 steps vs PS2 contacts"); plt.show()

In [ ]:
# PS3 bonus: missing / out-of-order steps against a reference, and two workers doing the same task
from handtrace.ps3 import compare_clips, check_sequence
if EXPECTED_STEPS:
    for c in ORDERED:
        acts = R.load_actions(OUT_ROOT, runner.stem_of(c))
        if acts:
            rep = check_sequence(acts, EXPECTED_STEPS)
            print(runner.stem_of(c), "complete & in order:", rep["complete_and_in_order"], "| missing:", rep["missing"],
                  "| out of order:", [x["expected_step"] for x in rep["out_of_order"]])
COMPARE = []                     # e.g. [("000060-010", "000060-015")]: two clips of the same task
for a, b in COMPARE:
    sa = next(runner.stem_of(c) for c in ORDERED if a in c); sb = next(runner.stem_of(c) for c in ORDERED if b in c)
    rows_cmp, summ = compare_clips(R.load_actions(OUT_ROOT, sa), R.load_actions(OUT_ROOT, sb), "A", "B")
    print(sa, "vs", sb, summ); display(pd.DataFrame(rows_cmp))

## 9 · PS5 · Video to insight

Four tools built only from the files above (CPU, minutes for all clips), each answering a factory question:

| tool | question it answers | output |
|---|---|---|
| quality filter | which clips / stretches are too blurry, dark or hand-less to use? | `ps5/quality.json`, `ps5_quality_all.csv` |
| safety monitor | when is a hand inside / near a hazard (spindle, press, conveyor…), and is the part moving? | `ps5/safety_events.csv` |
| plain-English search | "show every time someone uses a drill with the left hand" | `ps5_events_all.csv` + results table |
| dashboard | time per step type, tool use, hand utilisation, idle time, per clip | `ps5_dashboard.html` (numbers only) |
| training video | the key steps of a task, cut and captioned, with subtitles | private MP4 + `.srt` + steps CSV |

In [ ]:
from handtrace.ps5 import quality, safety, index, dashboard, training
from handtrace.ps5.search import search as search_events
if RUN_PS5:
    q_rows = quality.analyse_all(ORDERED, OUT_ROOT)
    display(pd.DataFrame(q_rows).set_index("clip")[["verdict", "why", "median_sharpness", "blurry_share", "dark_share", "hand_share", "bad_time_s"]])
    SCFG = safety.SafetyConfig(**({"hazard_words": HAZARD_WORDS} if HAZARD_WORDS else {}))
    s_rows = safety.analyse_all(ORDERED, OUT_ROOT, SCFG)
    if s_rows:
        display(pd.DataFrame(s_rows).sort_values(["risk", "duration_s"], ascending=[True, False]).head(30))

In [ ]:
# Plain-English search over PS1 + PS2 + PS3 + PS5 events (offline parser; the parsed filters are shown so results are explainable)
import ipywidgets as W
EVENTS = index.build_index(OUT_ROOT, STEMS)
print(len(EVENTS), "events | known objects:", ", ".join(index.known_objects(EVENTS)[:40]))
q_box = W.Text(value="show every time someone uses a screwdriver with the right hand", layout=W.Layout(width="70%"))
q_btn = W.Button(description="Search", button_style="primary"); q_out = W.Output()
def _run_search(_=None):
    with q_out:
        q_out.clear_output()
        res = search_events(EVENTS, q_box.value)
        f = {k: v for k, v in res["filters"].items() if v not in (None, [], "") and k != "query"}
        print("understood as:", f)
        if res["moments"]:
            display(pd.DataFrame(res["moments"]).assign(start=lambda d: d.t_start.map(lambda t: f"{int(t // 60)}:{t % 60:04.1f}")))
        else:
            print("no match. Try e.g.: 'left hand touching cloth', 'fasten longer than 5 s', 'no hands', 'unsafe', 'both hands after 1:00'")
q_btn.on_click(_run_search)
display(W.HBox([q_box, q_btn]), q_out); _run_search()

In [ ]:
# Look at the frames of a search result (start / middle / end, all layers): change MOMENT to inspect another row
from handtrace.review import ClipView
from handtrace.wilor.visualization import contact_sheet
MOMENT = 0
_res = search_events(EVENTS, q_box.value)["moments"]
if _res:
    m = _res[min(MOMENT, len(_res) - 1)]
    v = ClipView(next(c for c in ORDERED if runner.stem_of(c) == m["clip"]), OUT_ROOT)
    tiles = [v.compose(v.f_of(t), width=640) for t in (m["t_start"], (m["t_start"] + m["t_end"]) / 2, m["t_end"])]
    plt.figure(figsize=(18, 6)); plt.imshow(contact_sheet([t for t in tiles if t is not None])[:, :, ::-1]); plt.axis("off")
    plt.title(f"{m['clip']} {m['t_start']}-{m['t_end']} s: {m['what']}"); plt.show(); v.close()

In [ ]:
# Dashboard (numbers only) + tables
if RUN_PS5:
    DASH = dashboard.write_html(OUT_ROOT, STEMS)
    rows_d, verb_time, tool_use = dashboard.build_tables(OUT_ROOT, STEMS)
    display(pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows_d]).set_index("clip"))
    vt = pd.DataFrame(verb_time).set_index("clip"); vt = vt.loc[:, vt.sum() > 0]
    if not vt.empty:
        vt.plot.barh(stacked=True, figsize=(14, 0.5 + 0.45 * len(vt)), colormap="tab20", title="time per step type (s)").legend(
            bbox_to_anchor=(1, 1)); plt.show()
    print("open / download:", DASH)
    display(HTML(f"<iframe srcdoc='{open(DASH).read().replace(chr(39), '&#39;')}' style='width:100%;height:720px;border:0'></iframe>"))

In [ ]:
# Training video for one clip: key steps cut and captioned (private footage: event Drive folder only)
TRAIN_CLIP = ORDERED[0]
if RUN_PS5 and R.load_actions(OUT_ROOT, runner.stem_of(TRAIN_CLIP)):
    TRAIN = training.make_training_video(TRAIN_CLIP, OUT_ROOT, f"{WORK}/demo/{runner.stem_of(TRAIN_CLIP)}_training.mp4",
                                         min_step_s=1.0, max_total_s=60, width=1280)
    display(pd.DataFrame(TRAIN["steps"]))

## 10 · Manual video check (PS1 + PS2 + PS3 + PS5 on the original footage)

Three ways to check results by eye, all reading the same files:

1. **Frame inspector** (next cell, in the notebook). Scrub or play any clip with every layer drawn; toggle layers; show **raw vs post-processed
   PS1 side by side**; jump to the next *hard* moment (low score, interpolated, side corrected, two hands overlapping, frame edge, grab, release,
   PS3 step start, edge moved by PS2, safety, bad quality); and audit frames as you go.
2. **HTML review player** (cell after). One file per clip that holds **numbers only**. Open it on your laptop and pick the clip from disk: the
   overlay is drawn live over the original video at full frame rate, with a clickable PS1/PS2/PS3/PS5 timeline, frame stepping
   (`←`/`→`), speed control and the same audit form. Export the audit CSV and import it below. Teammates can audit in parallel.
3. **Rendered demo videos** (section 11): the brief's demo clips with every layer burned in, plus PS1 before / after.

**Audit protocol** (gives the measured numbers judges ask for): *Draw audit sample* picks ~100 frames per clip, 60 % hard moments + 40 % random.
For each frame count, **among the worker's own visible hands**: missed, false, left/right swapped, keypoints clearly off; and mark the PS2 contact /
PS3 step / safety flag shown as ok / wrong / na. `c` (or *All correct + next*) is one key press per correct frame.

In [ ]:
# 1 · Frame inspector (frames shown are company footage: clear this output before sharing the notebook)
from handtrace.review_widgets import inspector
INSPECT = inspector(ORDERED, OUT_ROOT, width=960, audit_samples=100)

In [ ]:
# 2 · HTML review players (numbers only; the reviewer opens the clip from their own disk)
from handtrace.review import ClipView, sample_audit_frames
from handtrace.review_html import export_review_html
PLAYER_CLIPS = [c for c in ORDERED if is_demo(c)] or ORDERED[:1]
for c in PLAYER_CLIPS:
    v = ClipView(c, OUT_ROOT)
    if not v.samples:
        sample_audit_frames(v, 100)              # the same stratified sample as the widget
    p = export_review_html(c, OUT_ROOT, view=v); v.close()
    print(f"{os.path.basename(c)} -> {p} ({os.path.getsize(p) / 1e6:.1f} MB)")
print("Download the HTML (Output panel), open it in Chrome/Edge, click 'Open the clip...' and choose the same video file.")

# Import audits exported from the player (upload the CSVs as a private dataset or to /kaggle/working)
from handtrace.review import AuditLog
for f in glob.glob("/kaggle/input/**/*_manual_audit.csv", recursive=True) + glob.glob(f"{WORK}/*_manual_audit.csv"):
    stem = os.path.basename(f)[: -len("_manual_audit.csv")]
    if stem in STEMS:
        print("imported", AuditLog(OUT_ROOT, stem).merge_csv(f), "rows from", f)

## 11 · Demo videos (private: event Drive folder only)

For each demo clip (or the first clip if `DEMO_CLIPS` is empty): **all layers** (PS1 hands, PS2 contact object, PS3 step caption, PS5 safety,
scrolling timeline) and **PS1 raw vs post-processed** side by side. Pick hard moments with `DEMO_START_S` (the inspector's jump list finds them).
The brief allows **2 minutes in total** across all demo videos.

In [ ]:
from handtrace.review import render_review_segment
DEMO_SEGMENT_S = 15          # seconds rendered per clip and video
DEMO_START_S = {}            # per clip: {"000003-004": 35} starts that clip's segment at 35 s
DEMO_DIR = f"{WORK}/demo"; os.makedirs(DEMO_DIR, exist_ok=True)
targets = [c for c in ORDERED if is_demo(c)] or ORDERED[:1]
for clip in targets:
    stem = runner.stem_of(clip); t0 = next((v for k, v in DEMO_START_S.items() if k in stem), 0)
    v = ClipView(clip, OUT_ROOT)
    render_review_segment(v, f"{DEMO_DIR}/{stem}_all_layers.mp4", t0, t0 + DEMO_SEGMENT_S, width=1280)
    render_review_segment(v, f"{DEMO_DIR}/{stem}_ps1_raw_vs_final.mp4", t0, t0 + DEMO_SEGMENT_S, layers={"ps1", "hud"},
                          width=1920, compare_raw=True)
    v.close()
total_s = len(targets) * DEMO_SEGMENT_S * 2
print(sorted(os.listdir(DEMO_DIR)), f"| {total_s} s of demo video" + ("  (over the 120 s limit: shorten DEMO_SEGMENT_S or drop one)" if total_s > 120 else ""))
_first = f"{DEMO_DIR}/{runner.stem_of(targets[0])}_all_layers.mp4"
if os.path.exists(_first):
    display(Video(_first, embed=True, width=960))   # private footage: clear before sharing

## 12 · Accuracy (manual audit + optional keypoint labels)

The **audit report** turns the reviewed frames into hand recall, precision, left/right accuracy, keypoint-OK rate and the share of fully correct
frames, plus PS2 / PS3 / safety accuracy where judged, always with the frame counts. Report the `test` split (alternate frames) if you tuned
post-processing on `dev`. If you also clicked exact keypoints with a labelling tool, `eval/metrics.py` scores raw vs final PCK / pixel error.

In [ ]:
from handtrace.review import audit_report
ACC = audit_report(OUT_ROOT, STEMS)
if ACC["total"]:
    display(pd.DataFrame([ACC["total"]], index=["all clips"]).T)
    if len(ACC["clips"]) > 1:
        display(pd.DataFrame(ACC["clips"]).set_index("clip"))
    for sp in ("dev", "test"):
        r = audit_report(OUT_ROOT, STEMS, split=sp)["total"]
        if r:
            print(sp, {k: r[k] for k in ("frames_reviewed", "hand_recall", "hand_precision", "side_accuracy", "keypoint_ok_rate")})
    json.dump(ACC, open(f"{OUT_ROOT}/accuracy_report.json", "w"), indent=1)
else:
    print("no audited frames yet: use the inspector (section 10) or import CSVs from the HTML player")

LABELS_DIR = None            # e.g. "/kaggle/input/<PRIVATE-LABELS>" with eval/labels/<clip>.json (exact keypoints)
CONTACT_SHEET = None         # e.g. "/kaggle/input/<PRIVATE-LABELS>/<clip>_contacts.csv" once filled in
if LABELS_DIR:
    for name in ("hands.jsonl", "hands_final.jsonl"):
        print("==", name); sh(f"cd {CODE} && python eval/metrics.py --labels-dir {LABELS_DIR} --pred-root {OUT_ROOT} --pred-name {name}")
ps2_dir = f"{OUT_ROOT}/{runner.stem_of(ORDERED[0])}/ps2_final"
if os.path.exists(f"{ps2_dir}/contacts.jsonl"):
    sh(f"cd {CODE} && python eval/contact_metrics.py template --contacts {ps2_dir}/contacts.jsonl --n 50 --out {ps2_dir}/contact_annotation_template.csv")
if CONTACT_SHEET:
    sh(f"cd {CODE} && python eval/contact_metrics.py score --sheet {CONTACT_SHEET}")
# Re-tune contact thresholds on the cached detections in seconds (CPU only), then re-run the PS3 refinement + PS5 cells:
# runner.retune_contacts(S, min_contact_frames=7, release_grace_frames=5)

## 13 · Package results

`results.zip`: every clip's PS1 / PS2 / PS3 / PS5 results, the audit, the review players, the dashboard and the cross-clip
CSVs (**no video, no frames**). It is still derived from private footage: share it only with the team and the organisers.
The code is the GitHub repository this notebook cloned; nothing from `/kaggle/working` should be committed.

In [ ]:
print("results zip:", runner.package_results(OUT_ROOT, ORDERED, f"{WORK}/results.zip"))
print("environment:", json.load(open(f"{OUT_ROOT}/environment.json")))

## 14 · Data-use cleanup (when the event ends)

Download only what you may keep for judging (to the organisers' private Drive folder), then delete. This removes the working copies and every
file stored in Gemini file storage under your key. Then delete the private Kaggle datasets (clips, MANO, labels) and this notebook's saved
versions and outputs in the Kaggle UI.

In [ ]:
CLEANUP = False
if CLEANUP:
    if os.environ.get("GEMINI_API_KEY"):
        from handtrace import ps3
        ps3.cleanup_remote(PS3)                       # data-use rule: delete EVERY file stored in Gemini storage under this key
    shutil.rmtree(WORK, ignore_errors=True)
    for p in glob.glob("/kaggle/working/**/*.mp4", recursive=True):
        os.remove(p)
    print("working copies removed; now delete the private datasets and notebook versions in the Kaggle UI")